Import Library

In [19]:
import pandas as pd
import numpy as np

Load Dataset & Cek Awal

In [20]:
# Membaca file Excel yang sudah di-upload
df = pd.read_excel('/dataset_kotor.xlsx')

print("=== DATA SEBELUM DIBERSIHKAN ===")
print(f"Jumlah data awal: {df.shape[0]} baris")
df.head()

=== DATA SEBELUM DIBERSIHKAN ===
Jumlah data awal: 30 baris


,ID,Nama,Usia,Jenis_Kelamin,Kota,Pendapatan,Pekerjaan,Rating_Kepuasan,Tanggal_Transaksi,Status_Member
0,1,Ahmad Fauzi,58.0,L,Jakarta,7614226,Karyawan Swasta,4,2026-01-15,Gold
1,2,Siti Aminah,33.0,P,Surabaya,4719583,PNS,5,16/01/2026,Silver
2,3,Budi Santoso,52.0,L,Jogja,10078673,Wiraswasta,3,2026-01-17,Bronze
3,4,Dewi Lestari,NaN,L,Jakarta,4571945,karyawan,NaN,2026-01-18,gold
4,5,Rian Hidayat,31.0,P,Yogyakarta,13100198,PNS,5,19-01-2026,Silver


Hapus Duplikasi

In [21]:
# Cek dan hapus data yang duplikat (Baris 11 dan 12)
print(f"Jumlah duplikat: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
print(f"Jumlah data setelah hapus duplikat: {df.shape[0]} baris")

Jumlah duplikat: 0
Jumlah data setelah hapus duplikat: 30 baris


 Rapikan Teks (Standarisasi)


In [22]:
# Merapikan Jenis_Kelamin (Menjadi 'L' dan 'P' saja)
gender_mapping = {
    'L': 'L', 'l': 'L', 'Laki-laki': 'L',
    'P': 'P', 'p': 'P', 'Perempuan': 'P'
}
df['Jenis_Kelamin'] = df['Jenis_Kelamin'].map(gender_mapping)

# Merapikan Nama Kota
kota_mapping = {
    'Jakarta': 'Jakarta', 'JKT': 'Jakarta',
    'Surabaya': 'Surabaya', 'Sby': 'Surabaya',
    'Jogja': 'Yogyakarta', 'Yogyakarta': 'Yogyakarta',
    'Bandung': 'Bandung', 'bdg': 'Bandung',
    'Medan': 'Medan', 'MEDAN': 'Medan'
}
df['Kota'] = df['Kota'].map(kota_mapping)

# Merapikan Status Member dan Pekerjaan (Huruf besar di awal)
df['Status_Member'] = df['Status_Member'].str.title()
df['Pekerjaan'] = df['Pekerjaan'].str.title()

print("Standarisasi teks selesai!")

Standarisasi teks selesai!


Rapikan Angka & Hapus Data Tidak Valid

In [23]:
# Usia: Hapus nilai tidak masuk akal (>100 dan <0)
df.loc[df['Usia'] > 100, 'Usia'] = np.nan
df.loc[df['Usia'] < 0, 'Usia'] = np.nan

# Pendapatan: Hapus tulisan "Rp " dan titik, ubah jadi angka
df['Pendapatan'] = df['Pendapatan'].astype(str).str.replace('Rp ', '', regex=False)
df['Pendapatan'] = df['Pendapatan'].str.replace('.', '', regex=False)
df['Pendapatan'] = pd.to_numeric(df['Pendapatan'], errors='coerce')

# Rating: Hapus tulisan "Bagus", nilai > 5, dan nilai < 1
df.loc[df['Rating_Kepuasan'] == 'Bagus', 'Rating_Kepuasan'] = np.nan
df.loc[df['Rating_Kepuasan'] > 5, 'Rating_Kepuasan'] = np.nan
df.loc[df['Rating_Kepuasan'] < 1, 'Rating_Kepuasan'] = np.nan
df['Rating_Kepuasan'] = pd.to_numeric(df['Rating_Kepuasan'], errors='coerce')

print("Pembersihan data numerik selesai!")

Pembersihan data numerik selesai!


Isi Data yang Kosong (Missing Values)

In [24]:
# Isi data teks yang kosong dengan Modus (data yang paling sering muncul)
df['Jenis_Kelamin'].fillna(df['Jenis_Kelamin'].mode()[0], inplace=True)
df['Kota'].fillna(df['Kota'].mode()[0], inplace=True)
df['Pekerjaan'].fillna(df['Pekerjaan'].mode()[0], inplace=True)

# Isi data angka yang kosong dengan Median
df['Usia'].fillna(df['Usia'].median(), inplace=True)
df['Pendapatan'].fillna(df['Pendapatan'].median(), inplace=True)
df['Rating_Kepuasan'].fillna(df['Rating_Kepuasan'].median(), inplace=True)

# Ubah Usia dan Rating kembali menjadi angka bulat (integer)
df['Usia'] = df['Usia'].astype(int)
df['Rating_Kepuasan'] = df['Rating_Kepuasan'].astype(int)

print("Pengisian data kosong selesai!")

Pengisian data kosong selesai!


/tmp/ipykernel_1538/4168854900.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Jenis_Kelamin'].fillna(df['Jenis_Kelamin'].mode()[0], inplace=True)
/tmp/ipykernel_1538/4168854900.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(valu

Rapikan Format Tanggal

In [25]:
# Ubah format tanggal. 'errors="coerce"' akan membuat tanggal salah (seperti 31 Feb) menjadi kosong
df['Tanggal_Transaksi'] = pd.to_datetime(df['Tanggal_Transaksi'], format='mixed', errors='coerce')

# Hapus baris yang tanggalnya error/kosong
df.dropna(subset=['Tanggal_Transaksi'], inplace=True)

# Samakan format tanggal menjadi YYYY-MM-DD
df['Tanggal_Transaksi'] = df['Tanggal_Transaksi'].dt.strftime('%Y-%m-%d')

print("Standarisasi tanggal selesai!")

Standarisasi tanggal selesai!


Hasil Akhir & Simpan File

In [26]:
print("=== DATA SETELAH DIBERSIHKAN ===")
print(f"Jumlah data akhir: {df.shape[0]} baris")
df.head()

# Simpan hasil cleaning ke file Excel baru
df.to_excel('dataset_bersih.xlsx', index=False)
print("\n✅ File 'dataset_bersih.xlsx' berhasil disimpan! ")

=== DATA SETELAH DIBERSIHKAN ===
Jumlah data akhir: 29 baris

✅ File 'dataset_bersih.xlsx' berhasil disimpan! 


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
